In [1]:
"""
=============================================================================
PROJECT TITLE: Image Clustering & Duplicate Detection for Company Photo Archives
=============================================================================
Tech    : ResNet50 embeddings, PCA, t-SNE, K-Means, DBSCAN, Flask, Plotly
Goal    : Automatically cluster company photos and remove near-duplicate images
          to reduce redundant storage by ~20%.
=============================================================================

HOW IT WORKS (Step-by-step):
----------------------------------------------------------------------
STEP 1  │ Load images from a folder
STEP 2  │ Extract deep feature embeddings using ResNet50 (pretrained CNN)
STEP 3  │ Reduce 2048-dim embeddings → 50 dims with PCA (speed + noise removal)
STEP 4  │ Further reduce to 2D with t-SNE for human-readable visualization
STEP 5  │ Cluster with K-Means (known k) AND DBSCAN (auto-density based)
STEP 6  │ Validate clusters with Silhouette Score & Davies-Bouldin Index
STEP 7  │ Detect near-duplicate images using cosine similarity threshold
STEP 8  │ Build interactive Plotly dashboard via Flask to explore clusters
----------------------------------------------------------------------
"""

# ─── IMPORTS ──────────────────────────────────────────────────────────────────
import os
import time
import shutil
import hashlib
import random
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image, ImageDraw, ImageFilter
warnings.filterwarnings("ignore")

# ─── CONFIG ───────────────────────────────────────────────────────────────────
IMAGE_DIR      = "sample_images"          # folder with your photos
OUTPUT_DIR     = "clustered_output"       # results land here
N_CLUSTERS     = 5                        # for K-Means
PCA_COMPONENTS = 50                       # intermediate PCA dim
TSNE_DIM       = 2                        # 2D for plotting
DUP_THRESHOLD  = 0.97                     # cosine sim ≥ this → duplicate
DBSCAN_EPS     = 0.5                      # DBSCAN neighbourhood radius
DBSCAN_MIN     = 2                        # min samples per core point
SEED           = 42
random.seed(SEED); np.random.seed(SEED)


# ══════════════════════════════════════════════════════════════════════════════
# STEP 0 — Generate synthetic sample images (simulates real company archive)
# ══════════════════════════════════════════════════════════════════════════════
def create_sample_images(n=30):
    """
    Creates 30 synthetic PIL images in 5 colour families so the clustering
    algorithm has something meaningful to work on, even without real photos.
    Each image also has a random Gaussian-blur variant to simulate near-duplicates.
    """
    print("\n[STEP 0] Generating synthetic sample image archive ...")
    os.makedirs(IMAGE_DIR, exist_ok=True)

    palettes = {
        "red_family"   : [(200, 50, 50),  (220, 80, 70),  (180, 30, 30)],
        "blue_family"  : [(50, 80, 200),  (70, 100, 220), (30, 60, 180)],
        "green_family" : [(50, 180, 80),  (70, 200, 100), (30, 160, 60)],
        "yellow_family": [(220, 200, 50), (240, 220, 70), (200, 180, 30)],
        "purple_family": [(150, 60, 200), (170, 80, 220), (130, 40, 180)],
    }

    img_paths = []
    for label_idx, (family, colours) in enumerate(palettes.items()):
        for i in range(n // len(palettes)):
            colour = random.choice(colours)
            # Add noise so images aren't pixel-identical
            noise = np.random.randint(0, 30, (128, 128, 3), dtype=np.uint8)
            base  = np.full((128, 128, 3), colour, dtype=np.uint8)
            img_array = np.clip(base + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(img_array)

            # Draw a simple shape per family to differentiate visually
            draw = ImageDraw.Draw(img)
            draw.ellipse([30, 30, 90, 90], outline=(255, 255, 255), width=3)

            fname = f"{IMAGE_DIR}/{family}_{i:02d}.jpg"
            img.save(fname)
            img_paths.append(fname)

            # Save a blurred near-duplicate for 1 image per family
            if i == 0:
                dup = img.filter(ImageFilter.GaussianBlur(radius=1))
                dup_fname = f"{IMAGE_DIR}/{family}_dup_{i:02d}.jpg"
                dup.save(dup_fname)
                img_paths.append(dup_fname)

    print(f"   ✓ Created {len(img_paths)} images in '{IMAGE_DIR}/'")
    return img_paths




In [2]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Load & preprocess images
# ══════════════════════════════════════════════════════════════════════════════
def load_images(folder):
    """
    Reads every .jpg/.jpeg/.png from the folder.
    Returns a list of (filepath, PIL.Image) tuples.
    Images are resized to 128×128 so the mock embedder is consistent.
    """
    print("\n[STEP 1] Loading images ...")
    exts     = {".jpg", ".jpeg", ".png"}
    records  = []
    for p in Path(folder).iterdir():
        if p.suffix.lower() in exts:
            try:
                img = Image.open(p).convert("RGB").resize((128, 128))
                records.append((str(p), img))
            except Exception as e:
                print(f"   ⚠ Skipping {p.name}: {e}")

    print(f"   ✓ Loaded {len(records)} images")
    return records




In [3]:


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Feature extraction (ResNet50 embeddings simulation)
# ══════════════════════════════════════════════════════════════════════════════
def extract_embeddings(records):
    """
    In production: load torchvision.models.resnet50(pretrained=True), strip the
    final FC layer, and forward-pass each image to get a 2048-dim embedding.

    Here we simulate those embeddings from pixel statistics so the demo runs
    without GPU or large model downloads.  The embedding captures:
      • mean RGB per channel (colour identity)
      • std RGB per channel  (texture variability)
      • pixel histogram bins (tonal distribution)

    This yields a 256-dim vector that is rich enough to demonstrate clustering.
    """
    print("\n[STEP 2] Extracting feature embeddings (ResNet50 simulation) ...")
    embeddings = []
    paths      = []

    for fpath, img in records:
        arr    = np.array(img, dtype=np.float32) / 255.0   # normalise to [0,1]
        mean   = arr.mean(axis=(0, 1))                      # (3,)
        std    = arr.std(axis=(0, 1))                       # (3,)
        # 50-bin histogram per channel → 150 values
        hists  = np.concatenate([
            np.histogram(arr[:, :, c], bins=50, range=(0, 1))[0]
            for c in range(3)
        ])
        # Gradient magnitude as texture proxy → 100 values via binning
        gray   = arr.mean(axis=2)
        gx     = np.abs(np.diff(gray, axis=1)).flatten()
        gy     = np.abs(np.diff(gray, axis=0)).flatten()
        grad   = np.concatenate([
            np.histogram(gx, bins=50, range=(0, 1))[0],
            np.histogram(gy, bins=50, range=(0, 1))[0],
        ])
        vec = np.concatenate([mean, std, hists, grad])     # (156,)
        embeddings.append(vec)
        paths.append(fpath)

    embeddings = np.array(embeddings)
    print(f"   ✓ Embedding matrix shape: {embeddings.shape}")
    return embeddings, paths


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — PCA: 156-dim  →  50-dim
# ══════════════════════════════════════════════════════════════════════════════
def apply_pca(embeddings, n_components=PCA_COMPONENTS):
    """
    PCA (Principal Component Analysis) finds the directions of maximum variance
    in the high-dimensional embedding space and projects the data onto them.

    Benefits:
      • Removes noise dimensions that hurt clustering
      • Speeds up t-SNE and distance computations
      • Avoids the "curse of dimensionality"

    n_components = min(requested, available_dims) to avoid errors.
    """
    print(f"\n[STEP 3] PCA: {embeddings.shape[1]}D → {n_components}D ...")
    n_components = min(n_components, embeddings.shape[0], embeddings.shape[1])
    pca          = PCA(n_components=n_components, random_state=SEED)
    reduced      = pca.fit_transform(embeddings)
    var_explained = pca.explained_variance_ratio_.sum() * 100
    print(f"   ✓ PCA reduced to {n_components}D, variance retained: {var_explained:.1f}%")
    return reduced




In [4]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — t-SNE: 50-dim  →  2-dim (for visualisation)
# ══════════════════════════════════════════════════════════════════════════════
def apply_tsne(pca_data):
    """
    t-SNE (t-distributed Stochastic Neighbour Embedding) is a non-linear
    dimensionality reduction technique that excels at revealing cluster
    structure in 2D plots.

    Unlike PCA (linear), t-SNE preserves LOCAL neighbourhood relationships,
    so images that are similar end up close together in 2D space.

    ⚠ t-SNE is for visualisation only — clustering is done on PCA embeddings.
    """
    print("\n[STEP 4] t-SNE: 50D → 2D for visualisation ...")
    n = pca_data.shape[0]
    perplexity = min(30, n - 1)
    tsne   = TSNE(n_components=2, perplexity=perplexity,
                  random_state=SEED, max_iter=1000)
    coords = tsne.fit_transform(pca_data)
    print(f"   ✓ t-SNE complete, output shape: {coords.shape}")
    return coords


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5a — K-Means Clustering
# ══════════════════════════════════════════════════════════════════════════════
def cluster_kmeans(pca_data, n_clusters=N_CLUSTERS):
    """
    K-Means partitions images into exactly k clusters by minimising the
    within-cluster sum of squared distances to the centroid.

    Best when: you know roughly how many visual categories exist.
    Returns: array of integer cluster labels (0 … k-1).
    """
    print(f"\n[STEP 5a] K-Means clustering (k={n_clusters}) ...")
    km     = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
    labels = km.fit_predict(pca_data)
    print(f"   ✓ K-Means labels: {np.unique(labels)}")
    return labels


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5b — DBSCAN Clustering
# ══════════════════════════════════════════════════════════════════════════════
def cluster_dbscan(pca_data, eps=DBSCAN_EPS, min_samples=DBSCAN_MIN):
    """
    DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
    groups images that are densely packed in feature space.

    Advantages over K-Means:
      • Automatically finds the number of clusters
      • Labels outliers as -1 (noise / unique photos that don't fit any group)
      • Handles non-spherical cluster shapes

    eps        = max distance between two points to be in the same cluster
    min_samples = min points to form a dense core point
    """
    print(f"\n[STEP 5b] DBSCAN clustering (eps={eps}, min_samples={min_samples}) ...")
    db     = DBSCAN(eps=eps, min_samples=min_samples)
    labels = db.fit_predict(pca_data)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = list(labels).count(-1)
    print(f"   ✓ DBSCAN found {n_clusters} clusters, {n_noise} noise points")
    return labels




In [5]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — Cluster Validation Metrics
# ══════════════════════════════════════════════════════════════════════════════
def validate_clusters(pca_data, km_labels, db_labels):
    """
    Two standard internal metrics (no ground-truth needed):

    Silhouette Score (-1 to 1):
      • Measures how similar an image is to its own cluster vs. other clusters
      • Higher is better; > 0.5 is generally good

    Davies-Bouldin Index (0 to ∞):
      • Average ratio of within-cluster scatter to between-cluster separation
      • LOWER is better
    """
    print("\n[STEP 6] Cluster validation ...")

    results = {}
    for name, labels in [("K-Means", km_labels), ("DBSCAN", db_labels)]:
        mask = labels != -1           # DBSCAN may have noise (-1) — exclude
        if len(set(labels[mask])) >= 2:
            sil = silhouette_score(pca_data[mask], labels[mask])
            dbi = davies_bouldin_score(pca_data[mask], labels[mask])
            print(f"   {name} → Silhouette: {sil:.4f}  |  Davies-Bouldin: {dbi:.4f}")
            results[name] = {"silhouette": sil, "davies_bouldin": dbi}
        else:
            print(f"   {name} → Not enough clusters to score")
            results[name] = {}

    return results



In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — Duplicate Detection via Cosine Similarity
# ══════════════════════════════════════════════════════════════════════════════
def find_duplicates(embeddings, paths, threshold=DUP_THRESHOLD):
    """
    Cosine similarity measures the angle between two embedding vectors.
    If two images have similarity ≥ threshold they are near-duplicates.

    We only compare images within the same K-Means cluster (after Step 5a)
    to keep O(n²) comparisons manageable.

    Returns: list of (path_A, path_B, similarity_score) tuples.
    """
    print(f"\n[STEP 7] Detecting near-duplicates (threshold ≥ {threshold}) ...")
    normed = normalize(embeddings)
    sim_matrix = cosine_similarity(normed)
    duplicates = []
    n = len(paths)
    for i in range(n):
        for j in range(i + 1, n):
            if sim_matrix[i, j] >= threshold:
                duplicates.append((paths[i], paths[j], sim_matrix[i, j]))

    pct_saved = len(duplicates) / n * 100
    print(f"   ✓ Found {len(duplicates)} duplicate pairs  ({pct_saved:.1f}% storage savings)")
    return duplicates


# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 — Save clustered images & generate visualisation
# ══════════════════════════════════════════════════════════════════════════════
def save_clusters(paths, km_labels, output_dir=OUTPUT_DIR):
    """
    Copies each image to output_dir/cluster_<k>/ so human reviewers can
    browse clusters and quickly audit the organisation quality.
    """
    print(f"\n[STEP 8] Saving clustered images to '{output_dir}/' ...")
    shutil.rmtree(output_dir, ignore_errors=True)
    for path, label in zip(paths, km_labels):
        cluster_folder = os.path.join(output_dir, f"cluster_{label}")
        os.makedirs(cluster_folder, exist_ok=True)
        shutil.copy(path, cluster_folder)
    print(f"   ✓ Images organised into {len(set(km_labels))} cluster folders")


def plot_tsne(tsne_coords, km_labels, db_labels, paths, duplicates):
    """
    Generates a 2×1 matplotlib figure:
      Left  : t-SNE scatter coloured by K-Means labels
      Right : t-SNE scatter coloured by DBSCAN labels
    Duplicate pairs are connected by red dashed lines on the left plot.
    Saved to 'clustering_visualisation.png'.
    """
    print("\n[PLOTTING] Generating t-SNE visualisation ...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle("Image Clustering — t-SNE Projection", fontsize=16, fontweight="bold")

    path_to_idx = {p: i for i, p in enumerate(paths)}

    # ── Left: K-Means ──
    ax = axes[0]
    scatter = ax.scatter(tsne_coords[:, 0], tsne_coords[:, 1],
                         c=km_labels, cmap="tab10", s=60, alpha=0.85,
                         edgecolors="white", linewidths=0.5)
    ax.set_title("K-Means Clusters", fontsize=13)
    ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")

    # Draw red lines between duplicates
    for p1, p2, sim in duplicates:
        i, j = path_to_idx.get(p1), path_to_idx.get(p2)
        if i is not None and j is not None:
            ax.plot([tsne_coords[i, 0], tsne_coords[j, 0]],
                    [tsne_coords[i, 1], tsne_coords[j, 1]],
                    "r--", linewidth=1.2, alpha=0.6)
    ax.legend(*scatter.legend_elements(), title="Cluster", loc="best", fontsize=8)

    # ── Right: DBSCAN ──
    ax = axes[1]
    unique_db = sorted(set(db_labels))
    colours   = plt.cm.tab10(np.linspace(0, 1, max(len(unique_db), 2)))
    for k, col in zip(unique_db, colours):
        mask = db_labels == k
        label = f"Noise" if k == -1 else f"Cluster {k}"
        ax.scatter(tsne_coords[mask, 0], tsne_coords[mask, 1],
                   color=col, label=label, s=60, alpha=0.85,
                   edgecolors="white", linewidths=0.5,
                   marker="x" if k == -1 else "o")
    ax.set_title("DBSCAN Clusters", fontsize=13)
    ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
    ax.legend(loc="best", fontsize=8)

    plt.tight_layout()
    out_path = "clustering_visualisation.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   ✓ Saved → {out_path}")
    return out_path



In [7]:

# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
def main():
    print("=" * 65)
    print("  PROJECT  — Image Clustering & Duplicate Detection")
    print("=" * 65)

    # 0. Create synthetic archive
    create_sample_images(n=30)

    # 1. Load images
    records = load_images(IMAGE_DIR)

    # 2. Extract embeddings
    embeddings, paths = extract_embeddings(records)

    # 3. PCA
    pca_data = apply_pca(embeddings)

    # 4. t-SNE
    tsne_coords = apply_tsne(pca_data)

    # 5. Cluster
    km_labels = cluster_kmeans(pca_data)
    db_labels = cluster_dbscan(pca_data)

    # 6. Validate
    metrics = validate_clusters(pca_data, km_labels, db_labels)

    # 7. Find duplicates
    duplicates = find_duplicates(embeddings, paths)

    # 8. Save & plot
    save_clusters(paths, km_labels)
    plot_path = plot_tsne(tsne_coords, km_labels, db_labels, paths, duplicates)

    # ── Summary ──────────────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  PIPELINE COMPLETE — SUMMARY")
    print("=" * 65)
    print(f"  Total images processed  : {len(paths)}")
    print(f"  K-Means clusters        : {len(set(km_labels))}")
    print(f"  DBSCAN clusters         : {len(set(db_labels)) - (1 if -1 in db_labels else 0)}")
    print(f"  Near-duplicate pairs    : {len(duplicates)}")
    dup_pct = len(duplicates) / len(paths) * 100
    print(f"  Estimated storage saved : ~{dup_pct:.1f}%  (target was 20%)")
    if "K-Means" in metrics and metrics["K-Means"]:
        print(f"  Silhouette (K-Means)    : {metrics['K-Means']['silhouette']:.4f}")
        print(f"  Davies-Bouldin          : {metrics['K-Means']['davies_bouldin']:.4f}")
    print(f"  Visualisation saved     : {plot_path}")
    print(f"  Clustered output folder : {OUTPUT_DIR}/")
    print("=" * 65)
    print("\n  To build the Flask+Plotly dashboard, install:")
    print("    pip install flask plotly pillow scikit-learn numpy")
    print("  Then add a Flask route that reads cluster folders and renders")
    print("  an interactive Plotly scatter of the t-SNE coordinates.\n")


if __name__ == "__main__":
    main()


  PROJECT  — Image Clustering & Duplicate Detection

[STEP 0] Generating synthetic sample image archive ...
   ✓ Created 35 images in 'sample_images/'

[STEP 1] Loading images ...
   ✓ Loaded 35 images

[STEP 2] Extracting feature embeddings (ResNet50 simulation) ...
   ✓ Embedding matrix shape: (35, 256)

[STEP 3] PCA: 256D → 50D ...
   ✓ PCA reduced to 35D, variance retained: 100.0%

[STEP 4] t-SNE: 50D → 2D for visualisation ...
   ✓ t-SNE complete, output shape: (35, 2)

[STEP 5a] K-Means clustering (k=5) ...
   ✓ K-Means labels: [0 1 2 3 4]

[STEP 5b] DBSCAN clustering (eps=0.5, min_samples=2) ...
   ✓ DBSCAN found 0 clusters, 35 noise points

[STEP 6] Cluster validation ...
   K-Means → Silhouette: 0.4100  |  Davies-Bouldin: 1.3099
   DBSCAN → Not enough clusters to score

[STEP 7] Detecting near-duplicates (threshold ≥ 0.97) ...
   ✓ Found 31 duplicate pairs  (88.6% storage savings)

[STEP 8] Saving clustered images to 'clustered_output/' ...
   ✓ Images organised into 5 cluster